# Analysing a RouteGuard run

This notebook loads a run directory written by `routeguard benchmark` and shows how to work with
the raw per-example records. It adds one analysis that is not part of the standard report:
**risk–coverage curves** of the initial-answer confidence for every system.

Set `RUN_DIR` to your run (by default the most recent run under `results/runs/`, falling back to a
fresh simulated smoke run so the notebook always executes).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from routeguard.evaluation.calibration import risk_coverage_curve
from routeguard.evaluation.report import load_records

ROOT = Path.cwd() if (Path.cwd() / "routeguard").exists() else Path.cwd().parent
os.chdir(ROOT)  # config and data paths are relative to the repository root
runs = sorted((ROOT / "results" / "runs").glob("*/processed/metrics.json"))
if runs:
    RUN_DIR = runs[-1].parent.parent
else:  # no run yet: create a small simulated one
    from routeguard.config import load_experiment_config
    from routeguard.experiments.runner import BenchmarkRunner

    cfg = load_experiment_config(ROOT / "experiments" / "smoke" / "smoke.yaml")
    cfg.pipeline["cache"] = {"enabled": False}
    RUN_DIR = BenchmarkRunner(cfg, output_dir=str(ROOT / "results" / "runs"), seeds=[0],
                              figures=False).run()
print("Run:", RUN_DIR)
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
print("status:", manifest["status"], "| simulated:", manifest["simulated"])

## Main results table (as written by the run)

In [ ]:
print((RUN_DIR / "tables" / "main_results.md").read_text())

## Per-language accuracy from the raw records

In [ ]:
records = load_records(sorted((RUN_DIR / "raw").glob("*.jsonl")))
systems = list(dict.fromkeys(r["system"] for r in records))
languages = sorted({r["language_variant"] for r in records})
header = f"{'system':28s}" + "".join(f"{lang:>10s}" for lang in languages)
print(header)
for s in systems:
    row = f"{s:28s}"
    for lang in languages:
        rs = [r["correct"] for r in records if r["system"] == s and r["language_variant"] == lang]
        row += f"{np.mean(rs):>10.3f}" if rs else f"{'-':>10s}"
    print(row)

## Risk–coverage curves

For each system, answers are sorted by initial-answer confidence (most confident first). The curve
shows the error rate among the accepted answers as coverage grows. Lower is better; a curve that
is flat at the overall error rate means the confidence carries no information.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for s in systems:
    rs = [r for r in records if r["system"] == s and r["attempts"][0]["confidence"] is not None]
    if len(rs) < 10:
        continue
    conf = np.array([r["attempts"][0]["confidence"] for r in rs])
    correct = np.array([r["initial_correct"] for r in rs])
    coverage, risk = risk_coverage_curve(conf, correct)
    ax.plot(coverage, risk, lw=1.5, label=s)
ax.set_xlabel("coverage (share of answers accepted)")
ax.set_ylabel("error rate among accepted")
ax.legend(fontsize=7, ncol=2)
ax.set_title("Risk–coverage of initial-answer confidence", loc="left")
if manifest["simulated"]:
    ax.text(0.5, 0.5, "SIMULATED", transform=ax.transAxes, alpha=0.2, fontsize=30,
            ha="center", color="red")
plt.show()